# Programmatic Schema Construction with SchemaBuilder

**ICBO 2025 Tutorial - Part 2**

This notebook demonstrates:
- Building schemas programmatically with SchemaBuilder
- Combining components from multiple sources
- Example: Composing a clinical research data harmonization schema

## Setup and Installation

First, install required packages:

In [1]:
# Uncomment and run if packages not installed
# !pip install linkml linkml-runtime biolink-model

In [2]:
# Import required libraries
from importlib.resources import files
from linkml.utils.schema_builder import SchemaBuilder
from linkml_runtime.utils.schemaview import SchemaView
from linkml_runtime.linkml_model import SlotDefinition
from linkml_runtime.dumpers import yaml_dumper
import warnings
warnings.filterwarnings('ignore')

---
# Programmatic Schema Construction with SchemaBuilder

**Why SchemaBuilder?**
- Build schemas programmatically in Python
- Combine components from multiple schemas
- Generate schemas from other data sources
- Implement schema transformations

**Implements the Builder Pattern:**
```python
sb = SchemaBuilder("my-schema")
sb.add_class(...).add_slot(...).add_enum(...)
schema = sb.schema
```

## SchemaBuilder Basics

Simple example of creating a schema from scratch:

In [3]:
# Create a new schema
sb = SchemaBuilder("sample-schema")

# Add slot definitions FIRST (with explicit properties)
sb.add_slot("name",
            description="Full name of the person",
            range="string")

sb.add_slot("age",
            description="Age in years",
            range="integer")

# Then add class that uses those slots
# Note: "email" will be auto-created with default range (string)
sb.add_class("Person",
             slots=["name", "age", "email"],
             description="A person")

# Get the schema
schema = sb.schema
print(f"Schema name: {schema.name}")
print(f"Classes: {list(schema.classes.keys())}")
print(f"Slots: {list(schema.slots.keys())}")

Schema name: sample-schema
Classes: ['Person']
Slots: ['name', 'age', 'email']


---
# Example: Building a Clinical Research Data Harmonization Schema

**Scenario:** You're a data architect for a multi-site clinical research network studying chronic diseases.

**Challenge:** Bridge biomedical knowledge graphs with clinical standards
- **Research sites** use Biolink Model for disease/drug entities
- **Clinical sites** use HL7/SNOMED standards
- **Need:** Unified schema supporting both research analysis AND clinical reporting

**Solution:** Compose a schema by combining:
- **Biolink** entity classes (diseases, drugs, phenotypes)
- **Clinical RO terms** not in Biolink (patient-level relationships)
- **Clinical valuesets** (HL7/SNOMED-mapped enums)

**Goal:** Enable harmonized data across molecular research and clinical care

## Step 1: Load Source Schemas

In [4]:
# Load Biolink Model (installed package)
biolink_yaml = files('biolink_model.schema') / 'biolink_model.yaml'
biolink_sv = SchemaView(str(biolink_yaml))

print(f"Loaded Biolink schema: {biolink_sv.schema.name}")
print(f"Biolink classes: {len(biolink_sv.all_classes())}")
print(f"Biolink slots: {len(biolink_sv.all_slots())}")

Loaded Biolink schema: Biolink-Model
Biolink classes: 308
Biolink slots: 474


In [5]:
# Create new schema with SchemaBuilder
sb = SchemaBuilder("clinical-research-schema")
sb.add_defaults()

# Add prefixes
sb.add_prefix("biolink", "https://w3id.org/biolink/vocab/")
sb.add_prefix("RO", "http://purl.obolibrary.org/obo/RO_")
sb.add_prefix("HL7", "http://terminology.hl7.org/CodeSystem/")
sb.add_prefix("SNOMED", "http://snomed.info/id/")

print("Created new schema: clinical-research-schema")

Created new schema: clinical-research-schema


## Step 2: Extract Classes and Slots from Biolink

Extract complete class definitions with all their slots

In [6]:
# Get relevant Biolink entity classes for clinical research
biolink_classes = ["disease", "drug", "phenotypic feature", "biological entity"]

print("Extracting Biolink classes with ALL their slots:")
for class_name in biolink_classes:
    cls = biolink_sv.get_class(class_name)
    if cls:
        # Use title case for our new schema
        new_class_name = class_name.title().replace(" ", "")

        # Get ALL slots for this class (including inherited)
        class_slot_names = biolink_sv.class_slots(class_name)

        # Extract and add complete slot definitions FIRST (preserves ALL properties)
        # IMPORTANT: Must add slots before adding class, otherwise SchemaBuilder
        # auto-creates empty slot stubs that block full definitions
        for slot_name in class_slot_names:
            if slot_name not in sb.schema.slots:
                slot = biolink_sv.get_slot(slot_name)
                
                # Handle the slot's range to avoid broken cross-schema references
                if slot.range:
                    if slot.range in biolink_sv.all_enums():
                        # Import enum if not already present
                        if slot.range not in sb.schema.enums:
                            sb.schema.enums[slot.range] = biolink_sv.get_enum(slot.range)
                        # Keep the enum range as-is
                    elif slot.range in biolink_sv.all_classes():
                        # Replace class ranges with string
                        slot.range = 'string'
                    elif biolink_sv.get_type(slot.range) is not None:
                        # Handle Biolink custom types - use base type if available
                        biolink_type = biolink_sv.get_type(slot.range)
                        if biolink_type.typeof:
                            slot.range = biolink_type.typeof
                        # else keep the type range as-is
                    # else: keep range as-is (likely already a built-in type)
                
                # Strip problematic cross-schema properties
                slot.is_a = None
                slot.domain = None
                slot.in_subset = None
                
                # Copy the entire slot definition - all 117+ properties preserved!
                sb.add_slot(SlotDefinition(**slot.__dict__))

        # NOW add the class with its slots (slots already have full metadata)
        sb.add_class(
            new_class_name,
            description=cls.description,
            slots=class_slot_names,
            exact_mappings=[f"biolink:{class_name.replace(' ', '')}"]
        )

        print(f"  ✓ {class_name} → {new_class_name} ({len(class_slot_names)} slots)")
    else:
        print(f"  ✗ {class_name} not found")

Extracting Biolink classes with ALL their slots:
  ✓ disease → Disease (14 slots)
  ✓ drug → Drug (21 slots)
  ✓ phenotypic feature → PhenotypicFeature (14 slots)
  ✓ biological entity → BiologicalEntity (14 slots)


## Step 3: Load Clinical ValueSets

Instead of creating simple enums, let's reuse standardized, ontology-grounded enums from the valuesets library with HL7 and SNOMED mappings.

In [7]:
# Load standardized clinical value sets from linkml/valuesets repository
vs_base = "https://raw.githubusercontent.com/linkml/valuesets/main/src/valuesets/schema/"
demographics_sv = SchemaView(vs_base + "demographics.yaml")
clinical_sv = SchemaView(vs_base + "medical/clinical.yaml")

# Extract ontology-grounded enums (HL7/SNOMED mapped)
education_level = demographics_sv.get_enum("EducationLevel")
blood_type = clinical_sv.get_enum("BloodTypeEnum")

# Add them to our schema
sb.schema.enums["EducationLevel"] = education_level
sb.schema.enums["BloodTypeEnum"] = blood_type

print(f"Added EducationLevel with {len(education_level.permissible_values)} values (HL7)")
print(f"Added BloodTypeEnum with {len(blood_type.permissible_values)} values (SNOMED)")

# Show example values with ontology mappings
print("\nExample EducationLevel values:")
for key in list(education_level.permissible_values.keys())[:3]:
    pv = education_level.permissible_values[key]
    meaning = getattr(pv, 'meaning', 'no mapping')
    print(f"  {key}: {pv.description} ({meaning})")

print("\nExample BloodTypeEnum values:")
for key in list(blood_type.permissible_values.keys())[:3]:
    pv = blood_type.permissible_values[key]
    meaning = getattr(pv, 'meaning', 'no mapping')
    print(f"  {key}: {pv.description} ({meaning})")

# Add custom type for dates (if not already present from Biolink extraction)
if "ISODateTime" not in sb.schema.types:
    sb.add_type("ISODateTime",
                typeof="string",
                description="ISO 8601 formatted datetime",
                pattern="^\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}")
    print("\nAdded ISODateTime type")
else:
    print("\nISODateTime type already exists (from Biolink)")

Added EducationLevel with 9 values (HL7)
Added BloodTypeEnum with 8 values (SNOMED)

Example EducationLevel values:
  ELEM: Elementary School (HL7:v3-EducationLevel#ELEM)
  SEC: Some secondary or high school education (HL7:v3-EducationLevel#SEC)
  HS: High School or secondary school degree complete (HL7:v3-EducationLevel#HS)

Example BloodTypeEnum values:
  A_POSITIVE: Blood type A, Rh positive (SNOMED:278149003)
  A_NEGATIVE: Blood type A, Rh negative (SNOMED:278152006)
  B_POSITIVE: Blood type B, Rh positive (SNOMED:278150003)

Added ISODateTime type


Now add custom slots and a class that uses these valuesets:

In [8]:
# Add custom slots for clinical observations (with enum ranges where appropriate)
sb.add_slot("education_level",
            description="Patient education level",
            range="EducationLevel")

sb.add_slot("blood_type",
            description="Patient blood type",
            range="BloodTypeEnum")

sb.add_slot("observation_date",
            description="Date of clinical observation",
            range="ISODateTime")

sb.add_slot("patient_id",
            description="Reference to the patient",
            range="BiologicalEntity")

print("Added custom slots")

# NOW add the class that uses these slots
sb.add_class("ClinicalObservation",
             description="A clinical observation capturing patient characteristics",
             slots=["id", "patient_id", "education_level",
                    "blood_type", "observation_date"])

print("Added ClinicalObservation class")

Added custom slots
Added ClinicalObservation class


## Step 4: Export and Introspect the Composed Schema

In [9]:
# Export to YAML file
schema = sb.schema
output_file = "clinical-research-schema.yaml"
yaml_dumper.dump(schema, output_file)
print(f"Exported schema to: {output_file}")

Exported schema to: clinical-research-schema.yaml


In [10]:
# Get the completed schema and create a SchemaView to introspect it
schema = sb.schema
composed_sv = SchemaView(schema)

print(f"\nCreated SchemaView for introspection:")
print(f"  Schema name: {schema.name}")
print(f"  Classes: {len(composed_sv.all_classes())}")
print(f"  Slots: {len(composed_sv.all_slots())}")
print(f"  Enums: {len(composed_sv.all_enums())}")


Created SchemaView for introspection:
  Schema name: clinical-research-schema
  Classes: 5
  Slots: 27
  Enums: 5


In [11]:
# Show the ontology-grounded clinical enums we added
print("EducationLevel values (with HL7 mappings):")
edu_enum = composed_sv.get_enum("EducationLevel")
for pv_name, pv_obj in list(edu_enum.permissible_values.items())[:5]:
    meaning = getattr(pv_obj, 'meaning', 'no mapping')
    print(f"  {pv_name}: {pv_obj.description}")
    print(f"    → {meaning}")

print(f"\nBloodTypeEnum has {len(composed_sv.get_enum('BloodTypeEnum').permissible_values)} blood types (SNOMED mapped)")

EducationLevel values (with HL7 mappings):
  ELEM: Elementary School
    → HL7:v3-EducationLevel#ELEM
  SEC: Some secondary or high school education
    → HL7:v3-EducationLevel#SEC
  HS: High School or secondary school degree complete
    → HL7:v3-EducationLevel#HS
  SCOL: Some College education
    → HL7:v3-EducationLevel#SCOL
  ASSOC: Associate's or technical degree complete
    → HL7:v3-EducationLevel#ASSOC

BloodTypeEnum has 8 blood types (SNOMED mapped)


In [12]:
# Examine the ClinicalObservation class we created
clin_obs_class = composed_sv.get_class("ClinicalObservation")
print(f"ClinicalObservation class:")
print(f"  Description: {clin_obs_class.description}")
print(f"  Slots: {clin_obs_class.slots}")

# Show the Biolink classes we extracted
print(f"\nBiolink entity classes in our schema:")
for cls_name in ["Disease", "Drug", "PhenotypicFeature", "BiologicalEntity"]:
    if cls_name in composed_sv.all_classes():
        print(f"  {cls_name}: {len(composed_sv.class_slots(cls_name))} slots")

# Show example slot with full metadata (including mappings)
print(f"\nExample slot with full metadata from Biolink:")
example_slot = composed_sv.get_slot("id")
print(f"  Slot: id")
print(f"    Description: {example_slot.description}")
print(f"    Range: {example_slot.range}")
print(f"    Required: {example_slot.required}")
if example_slot.exact_mappings:
    print(f"    Mappings: {example_slot.exact_mappings}")

ClinicalObservation class:
  Description: A clinical observation capturing patient characteristics
  Slots: ['id', 'patient_id', 'education_level', 'blood_type', 'observation_date']

Biolink entity classes in our schema:
  Disease: 14 slots
  Drug: 21 slots
  PhenotypicFeature: 14 slots
  BiologicalEntity: 14 slots

Example slot with full metadata from Biolink:
  Slot: id
    Description: A unique identifier for an entity. Must be either a CURIE shorthand for a URI or a complete URI
    Range: None
    Required: True
    Mappings: ['AGRKB:primaryId', 'gff3:ID', 'gpi:DB_Object_ID']


## Key Takeaway: SchemaBuilder → SchemaView Round-trip

Notice how we:
1. Used **SchemaView** to introspect existing schemas (Biolink, ValueSets)
2. Used **SchemaBuilder** to compose a new schema from multiple sources
3. Used **SchemaView** again to introspect our newly created schema

This round-trip pattern enables **iterative schema development** and validation!

---
## Summary

**SchemaBuilder:**
- Programmatically create and modify schemas
- Combine components from multiple sources
- Implement schema transformations
- Generate schemas from external data

**Key Points:**
- Define slots before classes that use them
- Use SchemaView to extract from existing schemas
- Use SchemaView to validate your built schema
- Powerful for composing domain-specific schemas from standard components

**In this example, we:**
- Extracted complete class definitions (with all slots) from Biolink
- Reused standardized HL7/SNOMED valuesets
- Created a harmonized clinical research schema bridging research and clinical domains